#### Imports

In [ ]:
# Make utils/ importable when running this notebook from notebooks/
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
# Standard library
import json
import os

# Third-party
import graphviz
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit

# Shared utils
from utils.checkpoint import Checkpoint
from utils.data_loading import load_gpkg, save_feature_list, save_model
from utils.plotting import plot_error_histogram, set_theme
from utils.stats import mixed_corr
from utils.train_config import CATEGORICAL_COLS, FEATURES, LABEL_COL, SPATIAL_CV_COL

## 1. Setup

In [ ]:
cp = Checkpoint("Start")

#### Paths and files

In [ ]:
# Setup path variable for graphviz binaries
os.environ["PATH"] += os.pathsep + "D:/.../Graphviz/bin"

In [ ]:
train_year = 2022

model_folder = "D:/.../model/"
validation_folder = "D:/.../validation/"
input_gpkg_file = "D:/.../EstIP_2022.gpkg"

model_path = model_folder + f"model_{train_year}.json"
features_path = model_folder + f"features_{train_year}.json"

#### Training parameters

In [ ]:
# Metrics
objective = "reg:squarederror"
eval_metric = "rmse"

In [ ]:
# Gridsearch
cv = 5

# Grid search runs on a random 10% sample of spatial blocks, keeping whole blocks together
search_sample_fraction = 0.1
search_random_state = 501

In [ ]:
# Training
train_size = 0.8
n_estimators = 50000

# SHAP values uses a random sample for effiency
shap_sample_size = 100_000
shap_random_state = 501

## 2. Data loading

In [ ]:
df = load_gpkg(input_gpkg_file, layer="s6_population_mapping")

In [ ]:
cp.mark("Data loaded")

## 3. Feature correlation

In [ ]:
df_corr = df[FEATURES]

In [ ]:
# Set correlation sample size
if len(df_corr) > 1_000_000:
    sample_size = 1_000_000
else:
    sample_size = len(df_corr)

correlations = mixed_corr(df_corr, CATEGORICAL_COLS, sample_size=sample_size)

In [ ]:
# Plot feature correlation
corr_matrix_png = validation_folder + f"feature_correlation_{train_year}.png"
plt.figure(figsize=(16, 12), dpi=150)
sns.set(font_scale=.5)
sns.heatmap(correlations, annot=True, fmt=".2f", linewidth=.5, cmap="coolwarm", vmin=-1, vmax=1)
plt.title(f"Feature correlation (Pearson / Cramérs V / Correlation ratio η) \n Sample size: {sample_size} \n [Train dataset {train_year}] \n", fontsize=9, color="0")
plt.savefig(corr_matrix_png, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
cp.mark("Feature correlation done")

## 4. Train/validation split

In [ ]:
X, y = df[FEATURES], df[LABEL_COL]
block_ids = df[SPATIAL_CV_COL]

# Split by whole spatial blocks
split_groups = block_ids.to_numpy(dtype=object).copy()
no_block = pd.isna(split_groups)
split_groups[no_block] = [f"__noblock_{i}" for i in np.flatnonzero(no_block)]

gss = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=501)
train_idx, val_idx = next(gss.split(X, y, groups=split_groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
block_ids_train, block_ids_val = block_ids.iloc[train_idx], block_ids.iloc[val_idx]

print(f"Train: {len(X_train)} rows ({len(X_train) / len(X):.1%}), "
      f"Val: {len(X_val)} rows ({len(X_val) / len(X):.1%})")
assert not (set(block_ids_train.dropna().unique()) & set(block_ids_val.dropna().unique())), \
    "a block spans both train and val"

In [ ]:
cp.mark("Train/validation split done")

## 5. Parameter grid search

In [ ]:
# Define basic XGB Regressor
gridsearch_regressor = xgb.XGBRegressor(device="cuda",
                                        tree_method="hist",
                                        enable_categorical=True,
                                        objective=objective,
                                        eval_metric=eval_metric)

In [ ]:
# Sample whole spatial blocks
# Buildings outside any block have NaN blck_id and can't be grouped for CV and are excluded from grid search
rng = np.random.default_rng(search_random_state)

unique_blocks = block_ids_train.dropna().unique()
n_sample_blocks = int(len(unique_blocks) * search_sample_fraction)
sampled_blocks = rng.choice(unique_blocks, size=n_sample_blocks, replace=False)

search_mask = block_ids_train.isin(sampled_blocks)
X_train_search = X_train[search_mask]
y_train_search = y_train[search_mask]
groups_search = block_ids_train[search_mask]

n_no_block = block_ids_train.isna().sum()
print(f"Grid search sample: {len(sampled_blocks)} of {len(unique_blocks)} blocks "
      f"({len(X_train_search)} of {len(X_train)} rows); "
      f"{n_no_block} rows with no block excluded from search only")

In [ ]:
# Group 1: learning_rate
gs_group_one = [
    {"learning_rate": [0.001], "n_estimators": [5000]},
    {"learning_rate": [0.01], "n_estimators": [2000]},
    {"learning_rate": [0.05], "n_estimators": [800]},
    {"learning_rate": [0.1], "n_estimators": [500]},
]
model_gridsearch_one = GridSearchCV(estimator=gridsearch_regressor, param_grid=gs_group_one,
                                     cv=GroupKFold(n_splits=cv),
                                     scoring="neg_root_mean_squared_error",
                                     refit=False, verbose=2)
model_gridsearch_one.fit(X_train_search, y_train_search, groups=groups_search, verbose=25)
optimized_group_one = model_gridsearch_one.best_params_

lr = [optimized_group_one["learning_rate"]]
n_est = [optimized_group_one["n_estimators"]]

In [ ]:
# Group 2: max_depth / min_child_weight
gs_group_two = {
    "learning_rate": lr,
    "max_depth": [3, 4, 5, 6, 7, 8],
    "min_child_weight": [5, 10, 50, 100, 250, 500, 1000],
    "n_estimators": n_est
}
model_gridsearch_two = GridSearchCV(estimator=gridsearch_regressor, param_grid=gs_group_two,
                                     cv=GroupKFold(n_splits=cv),
                                     scoring="neg_root_mean_squared_error",
                                     refit=False, verbose=2)
model_gridsearch_two.fit(X_train_search, y_train_search, groups=groups_search, verbose=25)
optimized_group_two = model_gridsearch_two.best_params_

# Named lookup, not positional (sklearn's dict ordering isn't a stable contract)
md = [optimized_group_two["max_depth"]]
mcw = [optimized_group_two["min_child_weight"]]

In [ ]:
# Group 3: colsample_bytree / subsample
gs_group_three = {
    "colsample_bytree": [0.5, 0.75, 1],
    "learning_rate": lr,
    "max_depth": md,
    "min_child_weight": mcw,
    "n_estimators": n_est,
    "subsample": [0.5, 0.75, 1]
}
model_gridsearch_three = GridSearchCV(estimator=gridsearch_regressor, param_grid=gs_group_three,
                                       cv=GroupKFold(n_splits=cv),
                                       scoring="neg_root_mean_squared_error",
                                       refit=False, verbose=2)
model_gridsearch_three.fit(X_train_search, y_train_search, groups=groups_search, verbose=25)
optimized_group_three = model_gridsearch_three.best_params_

ss = [optimized_group_three["subsample"]]
cbt = [optimized_group_three["colsample_bytree"]]

In [ ]:
# Group 4: gamma
gs_group_four = {
    "colsample_bytree": cbt,
    "gamma": [0, 0.01, 0.1, 1, 10],  # complexity control
    "learning_rate": lr,
    "max_depth": md,
    "min_child_weight": mcw,
    "n_estimators": n_est,
    "subsample": ss
}
model_gridsearch_four = GridSearchCV(estimator=gridsearch_regressor, param_grid=gs_group_four,
                                      cv=GroupKFold(n_splits=cv),
                                      scoring="neg_root_mean_squared_error",
                                      refit=False, verbose=2)
model_gridsearch_four.fit(X_train_search, y_train_search, groups=groups_search, verbose=25)
optimized_group_four = model_gridsearch_four.best_params_

ga = [optimized_group_four["gamma"]]

In [ ]:
# Group 5: alpha (L1) / lambda (L2)
gs_group_five = {
    "alpha": [0, 1, 5, 10],
    "colsample_bytree": cbt,
    "gamma": ga,
    "lambda": [0, 1, 5, 10],
    "learning_rate": lr,
    "max_depth": md,
    "min_child_weight": mcw,
    "n_estimators": n_est,
    "subsample": ss
}
model_gridsearch_five = GridSearchCV(estimator=gridsearch_regressor, param_grid=gs_group_five,
                                      cv=GroupKFold(n_splits=cv),
                                      scoring="neg_root_mean_squared_error",
                                      refit=False, verbose=2)
model_gridsearch_five.fit(X_train_search, y_train_search, groups=groups_search, verbose=25)
optimized_params = model_gridsearch_five.best_params_
print(optimized_params)

In [ ]:
# Export optimized parameters keyed by name
opt_params_json = model_folder + f"train_optimal_parameters_{train_year}.json"
with open(opt_params_json, "w", encoding="utf-8") as f:
    json.dump(optimized_params, f, indent=2)

In [ ]:
cp.mark("Grid search done")

## 6. Train model

In [ ]:
# Reload optimized parameters by key
with open(opt_params_json, "r", encoding="utf-8") as f:
    hp = json.load(f)

al = hp["alpha"]
cbt = hp["colsample_bytree"]
ga = hp["gamma"]
la = hp["lambda"]
lr = hp["learning_rate"]
md = int(hp["max_depth"])
ss = hp["subsample"]

# min_child_weight is an absolute rows-per-leaf count, found by searching on a
# search_sample_fraction slice of the training data. Scale it up so the same
# relative per-leaf restrictiveness holds once fit on the full X_train
mcw = hp["min_child_weight"] * (1 / search_sample_fraction)

# Early stopping patience scales with the chosen learning rate's own
# convergence timescale (its paired n_estimators from the search)
early_stopping_rounds = max(10, int(hp["n_estimators"] * 0.05))

In [ ]:
# Define and fit the model
eval_s = [(X_train, y_train), (X_val, y_val)]

model = xgb.XGBRegressor(device="cuda",
                          tree_method="hist",
                          enable_categorical=True,
                          objective=objective,
                          n_estimators=n_estimators,
                          early_stopping_rounds=early_stopping_rounds,
                          eval_metric=eval_metric,
                          max_depth=md,
                          min_child_weight=mcw,
                          colsample_bytree=cbt,
                          subsample=ss,
                          learning_rate=lr,
                          alpha=al,
                          gamma=ga,
                          reg_lambda=la)

model.fit(X_train, y_train, eval_set=eval_s, verbose=25)

In [ ]:
print("Optimal number of trees: " + str(model.best_iteration + 1))
print("Lowest validation RMSE: ", model.best_score)

In [ ]:
set_theme()

In [ ]:
# Plot and save training/validation curves
curves_png = model_folder + f"train_curves_{train_year}.png"
results = model.evals_result()
epochs = len(results["validation_0"][eval_metric])
x_axis = range(0, epochs)
fig, ax = plt.subplots(figsize=(8, 6), dpi=120)
ax.plot(x_axis, results["validation_0"][eval_metric], label="Train")
ax.plot(x_axis, results["validation_1"][eval_metric], label="Validation")
ax.legend()
plt.title("Train/validation learning curves \n", fontsize=12, color="0")
plt.xlabel("Number of trees", fontsize=12, color="0")
plt.ylabel(eval_metric, fontsize=12, color="0")
plt.xticks(fontsize=10, color="0.3")
plt.yticks(fontsize=10, color="0.3")
plt.grid(axis="y", alpha=0.75)
sns.despine()
plt.savefig(curves_png, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Save model and feature list
train_categories = {col: df[col].cat.categories.tolist()
                    for col in CATEGORICAL_COLS if col in df.columns}

save_model(model, model_path)
save_feature_list(FEATURES, CATEGORICAL_COLS, features_path,
                  best_iteration=model.best_iteration,
                  categories=train_categories)

print({col: len(vals) for col, vals in train_categories.items()})

In [ ]:
import importlib
import utils.data_loading
importlib.reload(utils.data_loading)
from utils.data_loading import load_gpkg, save_feature_list, save_model

In [ ]:
cp.mark("Model train")

## 7. Validation

In [ ]:
# Predict on validation data
predictions = model.predict(X_val)

prediction_table = pd.DataFrame({"Actuals": y_val,
                                 "Predictions": predictions,
                                 "Error": predictions - y_val,
                                 "AbsoluteError": abs(predictions - y_val),
                                 "RelativeError": (((predictions - y_val) / y_val) * 100)})

pred_table_csv = validation_folder + f"validation_predictions_{train_year}.csv"
prediction_table.to_csv(pred_table_csv, sep=";", index=False)
print(prediction_table.head())

In [ ]:
plot_error_histogram(prediction_table["Error"],
                     title="Histogram of Error \n (excluding 1st/99th percentiles) \n [Validation] \n",
                     xlabel="Error [inhabitants]",
                     out_path=validation_folder + f"validation_hist_error_{train_year}.png")

In [ ]:
plot_error_histogram(prediction_table["AbsoluteError"],
                     title="Histogram of Absolute Error \n (excluding 99th percentile) \n [Validation] \n",
                     xlabel="Absolute error [inhabitants]",
                     out_path=validation_folder + f"validation_hist_abserror_{train_year}.png",
                     percentiles=(None, 99))

In [ ]:
rel_prediction_table = prediction_table[prediction_table["Actuals"] > 0]

plot_error_histogram(rel_prediction_table["RelativeError"],
                     title="Histogram of Relative Error \n (excluding 1st/99th percentiles) \n [Validation] \n",
                     xlabel="Rel. Error [%]",
                     out_path=validation_folder + f"validation_hist_relerror_{train_year}.png")

In [ ]:
# Calculate statistics of actual/validation errors and save to CSV
statistics_table = pd.DataFrame({"Mean error": prediction_table["Error"].mean(),
                                 "Median error": prediction_table["Error"].median(),
                                 "Min error": prediction_table["Error"].min(),
                                 "Max error": prediction_table["Error"].max(),
                                 "Mean absolute error": prediction_table["AbsoluteError"].mean(),
                                 "Median absolute error": prediction_table["AbsoluteError"].median(),
                                 "Mean relative error": rel_prediction_table["RelativeError"].mean(),
                                 "Median relative error": rel_prediction_table["RelativeError"].median(),
                                 "Sum of actuals": prediction_table["Actuals"].sum(),
                                 "Sum of predictions": prediction_table["Predictions"].sum()}, index=[0])

statistics_table_csv = validation_folder + f"validation_statistics_{train_year}.csv"
statistics_table.to_csv(statistics_table_csv, sep=";", decimal=",", index=False)
print(statistics_table)

In [ ]:
# Plot and save feature importance
feature_imp_png = validation_folder + f"validation_xgb_feature_importance_{train_year}.png"
with sns.axes_style("white"):
    fig = plt.figure(figsize=(6, 6), dpi=130)
    ax = fig.add_subplot(111)
    xgb.plotting.plot_importance(model, ax=ax, height=0.5, importance_type="gain", color="purple", show_values=False)
    plt.title("Feature Importance \n XGBoost (Gain) \n", fontsize=12, color="0")
    plt.xlabel("Gain", fontsize=12, color="0")
    plt.ylabel("Features", fontsize=12, color="0")
    plt.xticks(fontsize=9, color="0.3")
    plt.yticks(fontsize=7, color="0.3")
    plt.grid(axis="x", alpha=0)
    plt.grid(axis="y", alpha=0.5, linestyle="dotted")
    sns.despine(left=True)
    plt.savefig(feature_imp_png, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
cp.report("Total training time")

## 8. SHAP values

In [ ]:
t_shap_start = cp.mark("Starting SHAP")

In [ ]:
# Convert model to native XGB API
booster = model.get_booster()

In [ ]:
# Sample the validation set for SHAP only
if len(X_val) > shap_sample_size:
    shap_rng = np.random.default_rng(shap_random_state)
    shap_idx = shap_rng.choice(len(X_val), size=shap_sample_size, replace=False)
    X_shap, y_shap = X_val.iloc[shap_idx], y_val.iloc[shap_idx]
else:
    X_shap, y_shap = X_val, y_val

print(f"SHAP on {len(X_shap)} of {len(X_val)} validation rows")

DXy_shap = xgb.DMatrix(X_shap, label=y_shap, enable_categorical=True)

In [ ]:
shap_and_base_values = booster.predict(DXy_shap, 
                                       pred_contribs=True, 
                                       iteration_range=(0, model.best_iteration + 1))
shap_values = shap_and_base_values[:, :-1]
base_values = shap_and_base_values[:, -1]

In [ ]:
feature_names = X_shap.columns.tolist()
shap_explanation = shap.Explanation(values=shap_values, 
                                    base_values=base_values,
                                    feature_names=feature_names, 
                                    data=X_shap.to_numpy())

In [ ]:
cp.mark("SHAP values computed")

In [ ]:
# SHAP feature importance
shap_fi_png = validation_folder + f"validation_shap_feature_importance_{train_year}.png"
with sns.axes_style("white"):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
    shap.plots.bar(shap_explanation.abs.mean(0), max_display=30, show=False)
    ax.set_title("Feature Importance \n SHAP-based [Validation]", fontsize=16)
    plt.xticks(color="0.3")
    plt.yticks(color="0.3")
    sns.despine(left=True)
    plt.savefig(shap_fi_png, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# SHAP feature importance in cohorts
shap_fc_png = validation_folder + f"validation_shap_feature_cohorts_{train_year}.png"
with sns.axes_style("white"):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
    shap.plots.bar(shap_explanation.cohorts(2).abs.mean(0), max_display=30, show=False)
    ax.set_title("Feature Importance (Cohorts) \n SHAP-based [Validation]", fontsize=16)
    plt.xticks(color="0.3")
    plt.yticks(color="0.3")
    sns.despine(left=True)
    plt.savefig(shap_fc_png, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# SHAP feature importance as beeswarm
shap_bs_png = validation_folder + f"validation_shap_beeswarm_{train_year}.png"
with sns.axes_style("white"):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
    shap.plots.beeswarm(shap_explanation, max_display=30, show=False)
    ax.set_title("Value-dependant Feature Importance \n SHAP-based [Validation]", fontsize=16)
    plt.xticks(color="0.3")
    plt.yticks(color="0.3")
    sns.despine(left=True)
    plt.savefig(shap_bs_png, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# SHAP scatterplot for the single most important feature by mean |SHAP| value
top_feature_idx = int(np.argmax(np.abs(shap_values).mean(axis=0)))
top_feature = feature_names[top_feature_idx]

shap_sc_png = validation_folder + f"validation_shap_scatter_{top_feature}_{train_year}.png"
with sns.axes_style("white"):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=120)
    shap.plots.scatter(shap_explanation[:, top_feature], dot_size=5, show=False, ax=ax)
    ax.set_title(f"Feature Scatterplot: {top_feature} \n SHAP-based [Validation]", fontsize=16)
    plt.xticks(color="0.3")
    plt.yticks(color="0.3")
    sns.despine(left=True)
    plt.savefig(shap_sc_png, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# SHAP feature importance for a single prediction
shap_sfi_png = validation_folder + f"validation_shap_single_fi_{train_year}.png"
with sns.axes_style("white"):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
    shap.plots.bar(shap_explanation[0], max_display=30, show=False)
    ax.set_title("Feature Importance (Single Prediction) \n SHAP-based [Validation]", fontsize=16)
    plt.xticks(color="0.3")
    plt.yticks(color="0.3")
    sns.despine(left=True)
    plt.savefig(shap_sfi_png, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# SHAP waterfall for a single prediction
shap_wf_png = validation_folder + f"validation_shap_waterfall_{train_year}.png"
with sns.axes_style("white"):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=88)
    shap.plots.waterfall(shap_explanation[0], max_display=30, show=False)
    ax.set_title("Decision Waterfall (Single prediction) \n SHAP-based [Validation]", fontsize=16)
    plt.xticks(color="0.3")
    plt.yticks(color="0.3")
    sns.despine(left=True)
    plt.savefig(shap_wf_png, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
cp.report("SHAP calculation time", since=t_shap_start)

#### Script runtime

In [ ]:
cp.finish()